# 00 · Setup: comprobar entorno y descargar datos

**Opcional**: cada notebook es independiente (su primera celda ⚙️ SETUP instala lo que falte y descarga sus
datos), así que puedes abrir cualquiera directamente, en local o en Colab. Este sirve para comprobar el entorno y
dejar todo descargado de antemano en `data/`:
- `yolo11n.pt`, `yolo11n-seg.pt`, `yolo11n-pose.pt` → modelos YOLO11 *nano* preentrenados en COCO
  (80 clases; la 0 es `person`). *Nano* = el más pequeño y rápido (n < s < m < l < x).
- `people.mp4` → vídeo de personas caminando (cámara fija, muchas oclusiones: perfecto para tracking).

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {"ultralytics": "ultralytics"}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
ROOT = ROOT or Path.cwd()   # fuera del repo (Colab): los datos se guardan en ./data
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "people.mp4").exists():   # vídeo de ejemplo (cámara fija, ~14 s, 1920×1080)
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/people-walking.mp4",
                               DATA / "people.mp4")
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
import sys, platform
from pathlib import Path


import torch, torchvision, cv2, numpy as np, ultralytics
print("Python      :", sys.version.split()[0], platform.machine())
print("PyTorch     :", torch.__version__, "| torchvision", torchvision.__version__)
print("OpenCV      :", cv2.__version__)
print("Ultralytics :", ultralytics.__version__)
print("MPS disponible:", torch.backends.mps.is_available())
print("CUDA disponible:", torch.cuda.is_available())

## Elegir dispositivo
Patrón estándar que usarás en todos los notebooks (y en código de producción):

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device()
x = torch.randn(1000, 1000, device=DEVICE)
print(DEVICE, (x @ x).shape)

## Descargar pesos YOLO11

In [ ]:
from ultralytics import YOLO

for name in ["yolo11n.pt", "yolo11n-seg.pt", "yolo11n-pose.pt"]:
    YOLO(str(DATA / name))  # si no existe, Ultralytics lo descarga a esa ruta
print(sorted(p.name for p in DATA.glob("*.pt")))

## Vídeo de ejemplo

In [ ]:
target = DATA / "people.mp4"   # ya lo descargó la celda SETUP
cap = cv2.VideoCapture(str(target))
print("frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
      "| fps:", cap.get(cv2.CAP_PROP_FPS),
      "| tamaño:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "x", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
cap.release()

## Prueba rápida: YOLO sobre un frame

In [ ]:
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(target))
ok, frame = cap.read()
cap.release()

model = YOLO(str(DATA / "yolo11n.pt"))
res = model(frame, device=DEVICE, verbose=False)[0]
print(f"{len(res.boxes)} detecciones; tiempos (ms):", {k: round(v, 1) for k, v in res.speed.items()})

plt.figure(figsize=(12, 7))
plt.imshow(res.plot()[:, :, ::-1])  # plot() devuelve BGR → lo pasamos a RGB para matplotlib
plt.axis("off");

Si ves cajas sobre las personas, **todo listo**. (Opcional) pon tu vídeo deportivo en `data/sport.mp4`.